# rhino-poc — Colab

**Baslamadan:** Runtime > Change runtime type > **GPU (T4)**.

Hucreleri **sirayla** kos. Atlama — 2. hucre degiskenleri tanimlar,
sonraki hucreler onlari kullanir (`NameError` alirsan 2. hucreyi kosmamissindir).

Colab burada sadece **GPU gereken adim** icin: COLMAP MVS. Bkz. `PLAN.md` SS4.

In [1]:
!nvidia-smi

Wed Aug 12 07:50:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Kurulum — repo + Drive + yollar
Tek hucre, her seyi tanimlar. Oturum dusarse **bu hucreden** devam et.

In [2]:
import os

REPO = '/content/rhino-poc'
WORK = '/content/work'          # calisma diski (HIZLI)

if os.path.isdir(REPO + '/.git'):
    !cd {REPO} && git fetch --quiet origin && git reset --hard origin/main
else:
    !rm -rf {REPO}
    !git clone --quiet https://github.com/Daml4Yilmaz/rhino-poc.git {REPO}

from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/rhino-poc-data'   # GIRDI: video/yakalama
SAVE = '/content/drive/MyDrive/rhino-poc-out'    # CIKTI: mesh buraya kaydedilir
os.makedirs(DATA, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

!cd {REPO} && git log --oneline -1
print('DATA icerigi:', os.listdir(DATA))

HEAD is now at 20733b9 Fail loudly on a missing capture path instead of falling through to ARKit
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
20733b9 (HEAD -> main, origin/main, origin/HEAD) Fail loudly on a missing capture path instead of falling through to ARKit
DATA icerigi: ['vaka_001', 'test.MOV']


**Neden `/content/work`?** COLMAP MVS on binlerce kucuk dosya yazar.
Drive FUSE uzerinden bu cok yavastir ve kopabilir. Hesaplama hizli diskte kosar,
**sonuclar** son hucrede Drive'a kopyalanir.

## 2. COLMAP (CUDA'li)
`%%bash` hucresi Python degiskenlerini GOREMEZ — burada yol yazmiyoruz, sorun degil.

In [3]:
%%bash
cd /opt
if [ ! -x /opt/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /opt/colmapenv/bin/colmap ]; then
  /opt/bin/micromamba create -y -q -p /opt/colmapenv -c conda-forge 'colmap=*=gpu*'
fi

In [3]:
import os
for c in ('/opt/colmapenv/bin/colmap', '/usr/local/colmap/bin/colmap'):
    if os.path.exists(c):
        os.system('ln -sf ' + c + ' /usr/local/bin/colmap')
        break
!colmap -h 2>&1 | head -3

COLMAP 3.11.1 -- Structure-from-Motion and Multi-View Stereo
(Commit Unknown on Unknown with CUDA)



Ciktida **`with CUDA`** yazmali. `without CUDA` yazarsa MVS adimi kosmaz.

**PATH uyarisi:** conda klasorunu PATH'in basina EKLEME — icindeki python sistem
python'unu golgeler ve `cv2` kirilir. Sadece symlink (ustteki hucre bunu yapiyor).

## 3. Python paketleri

In [4]:
# OpenCV KURMUYORUZ: Colab'da zaten var ve pipeline'in ihtiyacini karsiliyor.
!pip install -q open3d trimesh pycolmap typer pandas pillow

# --force-reinstall SART: pip ayni surumu 'already satisfied' deyip atlar
# ve repodaki yeni kod kurulmaz. Bu sessizce eski kodu kostururdu.
!pip install -q --force-reinstall --no-deps {REPO}

import importlib, poc
importlib.reload(poc)
MIN = (0, 4, 0)
got = tuple(int(x) for x in poc.__version__.split('.'))
print('poc surumu:', poc.__version__, '| dosya:', poc.__file__)
assert got >= MIN, (
    f'Kurulu surum {poc.__version__}, en az {".".join(map(str,MIN))} gerekli. '
    'Once 1. hucreyi (git reset) kos, sonra bu hucreyi. Hala eskiyse: '
    'Runtime > Restart session.')

import cv2
NEED = ['VideoCapture','imread','imwrite','resize','cvtColor','Laplacian',
        'SIFT_create','BFMatcher','findFundamentalMat']
missing = [n for n in NEED if not hasattr(cv2, n)]
print('cv2', getattr(cv2,'__version__','?'), '| eksik:', missing or 'yok')
if missing:
    print('\nCakisma: su ikisini kos, sonra Runtime > Restart session:')
    print("  !pip uninstall -y -q opencv-python opencv-contrib-python "
          "opencv-python-headless opencv-contrib-python-headless")
    print("  !pip install -q opencv-python-headless")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
                                                                                
 Usage: python -m poc.cli [OPTIONS] COMMAND [ARGS]...                           
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ process                                                                      │
│ scale     Sadece olcek: ARKit pozu + LiDAR -> scale.json                     │
│ measure   Sadece olcum: landmarks.json -> measurements.json                  │
╰──────────────────────────────────────────────────────

`poc` yerine **`python -m poc.cli`** kullaniyoruz: kurulan komut bazen Colab'in
PATH'ine girmez, modul cagrisi her zaman calisir.

**`-e` (editable) KULLANMA** — Jupyter ayni oturumda import edemez.
Repoda kod degisirse 1. ve bu hucreyi tekrar kos.

## 4A. TEST — duz .mov / .mp4

Videoyu Drive'a koy: `MyDrive/rhino-poc-data/test.mov`

Sadece **fotogrametri hattini** dogrular: kareler -> SfM -> MVS -> mesh.
**Model BIRIMSIZ olur** (`model_unitless.glb`) — aci ve Goode orani anlamli,
mm cinsinden uzunluk/genislik/sapma URETILEMEZ. Onun icin 4B gerekir.

Ilk denemede `--n-frames 150` birak: MVS kare sayisiyla dogru orantili,
150 kare T4'te ~15-25 dk, 300 kare bir saati asabilir.

In [ ]:
# Kernel yeniden baslarsa degiskenler silinir ve Colab '{CAPTURE}' gibi
# tanimsiz adlari DUZ METIN olarak komuta gecirir. Once burada dur.
for _v in ('DATA', 'WORK', 'SAVE'):
    assert _v in globals(), f'{_v} tanimsiz - once 1. KURULUM hucresini kos'

import os, glob

# Dosya adini ELLE YAZMA. Colab'da Drive mount'u BUYUK-KUCUK HARF DUYARLI
# ve iPhone videolari .MOV yazar; 'test.mov' yazinca dosya bulunamaz.
vids = [p for p in sorted(glob.glob(DATA + '/*'))
        if os.path.splitext(p)[1].lower() in ('.mov', '.mp4', '.m4v', '.avi')]
print('bulunan videolar:', [os.path.basename(v) for v in vids])
assert vids, 'DATA klasorunde video yok: ' + DATA + '  -> icerik: ' + str(os.listdir(DATA))

VIDEO = vids[0]          # birden fazlaysa indeksi degistir
OUT   = WORK + '/test_out'
print('kullanilan:', VIDEO)
!python -m poc.cli process '{VIDEO}' --out {OUT} --n-frames 150 --max-dim 1600

## 4B. GERCEK — Stray Scanner yakalamasi

Klasoru oldugu gibi Drive'a kopyala: `MyDrive/rhino-poc-data/<vaka>_stray/`
icinde `rgb.mp4`, `odometry.csv`, `camera_matrix.csv`, `depth/`, `confidence/`.

Adim adim kosuyoruz cunku `/content` gecicidir: SfM pahali, MVS ondan da
pahali. Once SfM, sonra yedek, sonra MVS. Boylece kopan bir baglanti en
fazla tek adimi goturur.

**`--resume` her hucrede var** ve zararsizdir: var olan ciktilari atlar,
yoksa hicbir sey yapmaz. Tek dikkat: `--n-frames` gibi bir parametreyi
DEGISTIRIRSEN eski kareler yeniden kullanilir — o zaman yeni bir `OUT` ver.

In [ ]:
# Kernel yeniden baslarsa degiskenler silinir ve Colab '{CAPTURE}' gibi
# tanimsiz adlari DUZ METIN olarak komuta gecirir. Once burada dur.
for _v in ('DATA', 'WORK', 'SAVE'):
    assert _v in globals(), f'{_v} tanimsiz - once 1. KURULUM hucresini kos'

CAPTURE = DATA + '/vaka_001_stray'    # Drive'daki gercek klasor adi
OUT     = WORK + '/vaka_001'

import os
assert os.path.isdir(CAPTURE), 'Yok: ' + CAPTURE + '  -> ' + str(os.listdir(DATA))
print('kaynak:', CAPTURE)
print('cikti :', OUT)

### 4B-1. SfM + hemen yedek

SfM biter bitmez sparse modeli Drive'a kopyalar (birkac MB, saniyeler).
Baglanti MVS sirasinda koparsa buraya geri donmek yerine dogrudan
4B-2'den devam edebilirsin.

In [ ]:
# Kernel yeniden baslarsa degiskenler silinir ve Colab '{CAPTURE}' gibi
# tanimsiz adlari DUZ METIN olarak komuta gecirir. Once burada dur.
for _v in ('DATA', 'WORK', 'SAVE'):
    assert _v in globals(), f'{_v} tanimsiz - once 1. KURULUM hucresini kos'

!python -m poc.cli process {CAPTURE} --out {OUT} --n-frames 300 --until sfm --resume

import os, shutil
bk = os.path.join(SAVE, os.path.basename(OUT) + '_sfm')
os.makedirs(bk, exist_ok=True)
for rel in ['frames_index.json', 'colmap/sparse', 'colmap/database.db']:
    src = os.path.join(OUT, rel)
    dst = os.path.join(bk, os.path.basename(rel))
    if os.path.isdir(src):
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
    elif os.path.exists(src):
        shutil.copy2(src, dst)
print('yedek ->', bk)
print(sorted(os.listdir(bk)))

### 4B-2. MVS + olcek + export (uzun adim)

T4'te 300 kare / 1600 px icin ~40-70 dk. Koparsa ayni hucreyi tekrar kos:
frames ve sfm atlanir, COLMAP yarim kalan derinlik haritalarini da atlar.

In [ ]:
# Kernel yeniden baslarsa degiskenler silinir ve Colab '{CAPTURE}' gibi
# tanimsiz adlari DUZ METIN olarak komuta gecirir. Once burada dur.
for _v in ('DATA', 'WORK', 'SAVE'):
    assert _v in globals(), f'{_v} tanimsiz - once 1. KURULUM hucresini kos'

!python -m poc.cli process {CAPTURE} --out {OUT} --n-frames 300 --resume

## 5. Sonuclari Drive'a kaydet
Yukarida hangi hucreyi kostuysan (`OUT` ondan gelir) bunu kos.

In [ ]:
import os, shutil, json

case = os.path.basename(OUT)
dst  = os.path.join(SAVE, case)
os.makedirs(dst, exist_ok=True)

KEEP = ['mesh_raw.ply', 'model.glb', 'model_unitless.glb',
        'scale.json', 'measurements.json', 'frames_index.json',
        'landmarks.json']
for f in KEEP:
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        shutil.copy2(p, dst)
        print('kaydedildi: %-22s %8.1f MB' % (f, os.path.getsize(p)/1e6))

print('\nDrive konumu:', dst)
sj = os.path.join(dst, 'scale.json')
if os.path.exists(sj):
    print(json.dumps(json.load(open(sj)), indent=2))

`mesh_raw.ply` ham yuzey, `model.glb` / `model_unitless.glb` goruntuleyici icin.
GLB'yi [gltf.report](https://gltf.report) veya Blender'da ac.
4B'de kafa bbox ~200-250 mm cikmali.

COLMAP ara ciktilari (`colmap/`, `frames/`) bilerek kopyalanmaz — GB'larca yer tutar.
Gerekirse: `!cp -r {OUT}/colmap {dst}/`

## 6. TESHIS — cekim fotogrametriye uygun mu?

SfM cokerse (kayit orani dusukse) bunu kos. ~2 dakika, GPU harcamaz.
Videoyu **kosmadan once** de kosabilirsin: uygun olmayan bir cekim icin
40 dakika beklemeye gerek yok.

In [ ]:
import os, cv2, numpy as np

# Stray Scanner klasoru verildiyse video ONUN ICINDEDIR. Elle VIDEO
# yazmak tehlikeli: 4A hucresinden kalan eski yol bellekte durur ve
# yanlis dosya sessizce analiz edilir.
SRC = globals().get('CAPTURE') or globals().get('VIDEO')
assert SRC, 'Once 4A veya 4B hucresini kos (VIDEO ya da CAPTURE tanimlansin)'
VID = os.path.join(SRC, 'rgb.mp4') if os.path.isdir(SRC) else SRC
print('analiz edilen:', VID)

c = cv2.VideoCapture(VID)
N = int(c.get(cv2.CAP_PROP_FRAME_COUNT))
fps = c.get(cv2.CAP_PROP_FPS) or 30
# Aralik, pipeline'in gercekte SECECEGI aralikla ayni olmali (~300 kare).
STEP = max(1, N // 300)
start = N // 3
want = set(range(start, min(start + STEP*30, N), STEP))
frames, i = [], 0
while True:
    ok, f = c.read()
    if not ok: break
    if i in want: frames.append(cv2.resize(f, (0,0), fx=0.5, fy=0.5))
    i += 1
c.release()
print(f'video: {N} kare @ {fps:.0f} fps = {N/fps:.0f} sn')
print(f'{len(frames)} ornek kare, aralik {STEP} kare = {STEP/fps:.3f} sn')

g = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
sift = cv2.SIFT_create(nfeatures=8000)
kd = [sift.detectAndCompute(x, None) for x in g]
nf = [len(k) for k, _ in kd]

bf = cv2.BFMatcher(cv2.NORM_L2)
inl, disp = [], []
for a in range(len(frames)-1):
    (k1,d1),(k2,d2) = kd[a], kd[a+1]
    if d1 is None or d2 is None or len(k1) < 8 or len(k2) < 8:
        inl.append(0); continue
    good = [x for x,y in bf.knnMatch(d1,d2,k=2) if x.distance < 0.75*y.distance]
    if len(good) < 8:
        inl.append(len(good)); continue
    p1 = np.float32([k1[x.queryIdx].pt for x in good])
    p2 = np.float32([k2[x.trainIdx].pt for x in good])
    F, m = cv2.findFundamentalMat(p1, p2, cv2.FM_RANSAC, 3.0)
    inl.append(int(m.sum()) if m is not None else 0)
    disp.append(float(np.median(np.linalg.norm(p1-p2, axis=1))))

feat, match, px = np.mean(nf), np.mean(inl), np.median(disp)
dt = STEP / fps
pxs = px / dt
print(f'\nozellik/kare      : {feat:7.0f}')
print(f'ic-eslesme/cift   : {match:7.0f}   (>50 iyi)')
print(f'kayma             : {px:7.1f} px / {dt:.3f} sn')
print(f'kayma hizi        : {pxs:7.0f} px/sn  (>60 iyi, <25 paralaks yok)')

# Esikler OLCUMLE kalibre edildi, tahminle degil:
#   test.mov      : feat 522, pxs  12 -> gercekte %2  kayit (COKTU)
#   Stray 4458ba  : feat 632, pxs 170 -> gercekte %81 kayit (CALISTI)
# Ikisinde de ozellik sayisi benzer; ayirt eden PARALAKS. Bu yuzden doku
# esigi cok dusuk tutuldu, karar esasen kayma hizina birakildi.
print()
if pxs < 25:
    print('RET: kamera yer degistirmemis. Paralaks yok -> SfM cozulemez.')
    print('     Sabit kameradan cekim ya da yerinde pan degil, denegin')
    print('     ETRAFINDA yurumek gerekiyor.')
elif pxs > 3000:
    print('RET: kareler arasi hareket cok fazla. Daha yavas cek.')
elif feat < 250:
    print('RET: neredeyse hic oznitelik yok (duz fon + asiri bulaniklik).')
elif match < 50:
    print('ZAYIF: ortusme dusuk. --n-frames artir (orn. 400).')
elif feat < 800:
    print('UYGUN (dusuk doku): kosulabilir ama fon dokusuz. Referans:')
    print('     632 ozellik/kare ile %81 kayit alindi. Daha dokulu fon,')
    print('     daha cok kaydedilmis kare demek.')
else:
    print('UYGUN: bu cekim fotogrametri icin makul.')